# Week 2: Data Quality Audit and Preprocessing

## 1. Objective and scope
This notebook documents Week 2: Data Collection, Cleaning, and Preprocessing for Logistics Analysis. It audits the supplied records, reuses the preprocessing functions, and explains how data quality affects logistics interpretation. It does not repeat Week 1 modeling, clustering, or optimization. All source rows and raw values are preserved; concerns are flagged rather than silently repaired or removed.

**Working-directory assumption:** run with the repository root (`logistics-route-optimization`) as the current directory. Paths are relative to that root. This notebook reads existing outputs and reproduces preprocessing in memory; it does not overwrite the raw CSV or Week 1 cleaned CSV.


In [1]:
from pathlib import Path
import json, sys
import pandas as pd
PROJECT_ROOT = Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "Zomato Dataset.csv"
WEEK1_PATH = PROJECT_ROOT / "data" / "cleaned_logistics_data.csv"
WEEK2_PATH = PROJECT_ROOT / "data" / "week2_cleaned_logistics_data.csv"
AUDIT_PATH = PROJECT_ROOT / "reports" / "week2_data_quality_summary.json"
required = [RAW_PATH, WEEK1_PATH, WEEK2_PATH, AUDIT_PATH, PROJECT_ROOT / "src" / "data_preprocessing.py"]
missing = [str(p.relative_to(PROJECT_ROOT)) for p in required if not p.exists()]
if missing: raise FileNotFoundError(f"Run from repository root; missing: {missing}")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data_preprocessing import (FIELD_DESCRIPTIONS, build_audit_summary, calculate_delivery_distance,
    clean_column_names, detect_duplicates, detect_iqr_outliers, normalize_categories,
    parse_order_and_pickup_times, parse_order_date, preprocess_dataset, validate_coordinates)
source = pd.read_csv(RAW_PATH, low_memory=False)
week2_saved = pd.read_csv(WEEK2_PATH, low_memory=False)
with AUDIT_PATH.open(encoding="utf-8") as f: audit = json.load(f)
print("Project:", PROJECT_ROOT)
print("Raw shape:", source.shape, "| Saved Week 2 shape:", week2_saved.shape)


Project: C:\Users\Kailash\Desktop\Documents\logistics-route-optimization
Raw shape: (45584, 20) | Saved Week 2 shape: (45584, 59)


## 2. Dataset description and data dictionary
The local dataset has 45,584 rows and 20 source columns. Dates parse using the observed day-month-year format and cover February 11 to April 6, 2022. The descriptions below are practical interpretations of column names and observed values, not a verified publisher data dictionary.


In [2]:
parsed_source_dates = pd.to_datetime(source["Order_Date"], format="%d-%m-%Y", errors="coerce")
print("Dimensions:", source.shape, "| parsed date range:", parsed_source_dates.min().date(), "to", parsed_source_dates.max().date())
print("Original data types:\n", source.dtypes.astype(str).to_string())
field_map = {"ID":"id", "Delivery_person_ID":"delivery_person_id", "Delivery_person_Age":"delivery_person_age",
"Delivery_person_Ratings":"delivery_person_ratings", "Restaurant_latitude":"restaurant_latitude",
"Restaurant_longitude":"restaurant_longitude", "Delivery_location_latitude":"delivery_location_latitude",
"Delivery_location_longitude":"delivery_location_longitude", "Order_Date":"order_date", "Time_Orderd":"time_orderd",
"Time_Order_picked":"time_order_picked", "Weather_conditions":"weather_conditions",
"Road_traffic_density":"road_traffic_density", "Vehicle_condition":"vehicle_condition",
"Type_of_order":"type_of_order", "Type_of_vehicle":"type_of_vehicle", "multiple_deliveries":"multiple_deliveries",
"Festival":"festival", "City":"city", "Time_taken (min)":"time_taken_min"}
dictionary = pd.DataFrame([{"Original field":c,"dtype":str(source[c].dtype),"Description":FIELD_DESCRIPTIONS[field_map[c]]} for c in source.columns])
print("\nData dictionary (all 20 source fields):\n", dictionary.to_string(index=False))


Dimensions: (45584, 20) | parsed date range: 2022-02-11 to 2022-04-06
Original data types:
 ID                                 str
Delivery_person_ID                 str
Delivery_person_Age            float64
Delivery_person_Ratings        float64
Restaurant_latitude            float64
Restaurant_longitude           float64
Delivery_location_latitude     float64
Delivery_location_longitude    float64
Order_Date                         str
Time_Orderd                        str
Time_Order_picked                  str
Weather_conditions                 str
Road_traffic_density               str
Vehicle_condition                int64
Type_of_order                      str
Type_of_vehicle                    str
multiple_deliveries            float64
Festival                           str
City                               str
Time_taken (min)                 int64

Data dictionary (all 20 source fields):
              Original field   dtype                                                   

## 3. Source and provenance
A Kaggle listing titled *Zomato Delivery Operations Analytics Dataset* is a candidate match only. The exact identity and provenance of this local CSV have not been verified. Do not infer that Zomato collected it. License, version, acquisition date, collection method, and original collector remain unknown/unverified. See [`data/DATASET_SOURCE.md`](../data/DATASET_SOURCE.md).

## 4. Initial quality assessment
The following checks use the untouched source CSV. Duplicate counts distinguish excess occurrences from all rows participating in a duplicate group.


In [3]:
missing_source = source.isna().sum()
print("Missing source values:\n", missing_source[missing_source.gt(0)].to_string())
print("Exact duplicate excess rows / participating rows:", int(source.duplicated().sum()), int(source.duplicated(keep=False).sum()))
print("Duplicate ID excess records / participating rows:", int(source["ID"].duplicated().sum()), int(source["ID"].duplicated(keep=False).sum()))
for c in ["Weather_conditions","Road_traffic_density","Type_of_order","Type_of_vehicle","Festival","City"]:
    print(f"\n{c} frequencies including missing:\n{source[c].value_counts(dropna=False).to_string()}")
nums=["Delivery_person_Age","Delivery_person_Ratings","Restaurant_latitude","Restaurant_longitude",
"Delivery_location_latitude","Delivery_location_longitude","Vehicle_condition","multiple_deliveries","Time_taken (min)"]
print("\nNumeric descriptive statistics:\n",source[nums].describe().round(3).to_string())


Missing source values:
 Delivery_person_Age        1854
Delivery_person_Ratings    1908
Time_Orderd                1731
Weather_conditions          616
Road_traffic_density        601
multiple_deliveries         993
Festival                    228
City                       1200
Exact duplicate excess rows / participating rows: 0 0
Duplicate ID excess records / participating rows: 0 0

Weather_conditions frequencies including missing:
Weather_conditions
Fog           7653
Stormy        7584
Cloudy        7533
Sandstorms    7494
Windy         7422
Sunny         7282
NaN            616

Road_traffic_density frequencies including missing:
Road_traffic_density
Low       15476
Jam       14139
Medium    10945
High       4423
NaN         601

Type_of_order frequencies including missing:
Type_of_order
Snack     11530
Meal      11456
Drinks    11321
Buffet    11277

Type_of_vehicle frequencies including missing:
Type_of_vehicle
motorcycle          26429
scooter             15273
electric_scoote

**Interpretation.** Missing values occur in age (1,854), ratings (1,908), order time (1,731), weather (616), traffic (601), multiple deliveries (993), festival (228), and City (1,200). No exact duplicate rows or duplicate IDs were found. `City` appears to be a broad area label; its exact semantics are undocumented.

## 5. Missing-value handling
Numeric missing values remain missing to avoid biased averages or invented measurements. Unknown categorical values are labeled `Unknown` in the processed file so they can be counted without dropping their records. The audit separately reports raw missing values and nonmissing strings that failed parsing.


In [4]:
# Use raw source headers when counting source missingness.
for c in ["Delivery_person_Age","Delivery_person_Ratings","Time_Orderd","Weather_conditions",
          "Road_traffic_density","multiple_deliveries","Festival","City"]:
    print(f"{c}: source missing = {int(source[c].isna().sum())}")
print("\nUnknown labels in processed categoricals:")
for c in ["weather_conditions","road_traffic_density","festival","city"]:
    print(c, int(week2_saved[c].astype("string").eq("Unknown").sum()))
print("\nSource missing / parse failure / resulting parsed missing:")
for raw, parsed in [("order_date","order_date_parsed"),("time_orderd","order_time_minutes"),("time_order_picked","pickup_time_minutes")]:
    diag = audit["date_parsing"] if raw == "order_date" else audit["time_parsing"][raw]
    print(raw, diag["source_missing_count"], diag["parse_failure_count"], int(week2_saved[parsed].isna().sum()))


Delivery_person_Age: source missing = 1854
Delivery_person_Ratings: source missing = 1908
Time_Orderd: source missing = 1731
Weather_conditions: source missing = 616
Road_traffic_density: source missing = 601
multiple_deliveries: source missing = 993
Festival: source missing = 228
City: source missing = 1200

Unknown labels in processed categoricals:
weather_conditions 616
road_traffic_density 601
festival 228
city 1200

Source missing / parse failure / resulting parsed missing:
order_date 0 0 0
time_orderd 1731 430 2161
time_order_picked 0 1320 1320


## 6. Date and time parsing
Dates use the observed `%d-%m-%Y` format. Valid 24-hour `HH:MM[:SS]` and unambiguous fractional-day values are parsed; raw strings stay in the output. `1` is ambiguous and `24:05:00` is outside a standard clock, so neither is silently repaired.

The retained overnight pickup heuristic is an **assumption**: if pickup clock time is earlier than order time, add one day. Intervals exceeding 12 hours are set missing and flagged. There is no pickup date or source convention to verify either rule.


In [5]:
normalized = clean_column_names(source)
with_dates, date_diag = parse_order_date(normalized)
with_times, time_diag = parse_order_and_pickup_times(with_dates)
print("Date parse summary:", {k:date_diag[k] for k in ["source_missing_count","parse_failure_count"]})
for raw in ["time_orderd","time_order_picked"]:
    d=time_diag[raw]
    print(raw, "source missing",d["source_missing_count"],"parse failures",d["parse_failure_count"],"examples",d["failure_examples"])
print("Pickup interval rule:",time_diag["order_to_pickup_interval_assumption"])


Date parse summary: {'source_missing_count': 0, 'parse_failure_count': 0}
time_orderd source missing 1731 parse failures 430 examples [{'source_row_index': 116, 'id': '0xbd03', 'time_orderd': '1'}, {'source_row_index': 599, 'id': '0xd152', 'time_orderd': '1'}, {'source_row_index': 696, 'id': '0x500f', 'time_orderd': '1'}, {'source_row_index': 825, 'id': '0x577c', 'time_orderd': '1'}, {'source_row_index': 886, 'id': '0xd1b8', 'time_orderd': '1'}]
time_order_picked source missing 0 parse failures 1320 examples [{'source_row_index': 66, 'id': '0x4b7f', 'time_order_picked': '24:05:00'}, {'source_row_index': 92, 'id': '0x3185', 'time_order_picked': '24:10:00'}, {'source_row_index': 116, 'id': '0xbd03', 'time_order_picked': '24:15:00'}, {'source_row_index': 155, 'id': '0x51e4', 'time_order_picked': '1'}, {'source_row_index': 157, 'id': '0x1dac', 'time_order_picked': '1'}]
Pickup interval rule: {'negative_clock_differences_rolled_forward': 0, 'intervals_over_12_hours_set_missing': 0, 'note': 

Order time has 1,731 source-missing values plus 430 parse failures, giving 2,161 missing parsed values. Pickup time has no source-missing values and 1,320 parse failures, including raw `1` and `24:05:00` values.

## 7. Coordinate validation
Global ranges (latitude -90 to 90; longitude -180 to 180) are checked separately from the broad assumed India bounds (latitude 6 to 38; longitude 68 to 98). Negative coordinates may be valid globally; they are not rejected merely for their sign. In this dataset they may be outside the assumed operating region. Zero, near-zero, nonfinite, out-of-range, and out-of-region concerns are flagged; no signs or values are silently repaired. Distance is calculated only for valid coordinate pairs and is straight-line Haversine distance, not road distance.


In [6]:
coords=validate_coordinates(normalized)
print("Global valid pairs:",int(coords.coordinates_globally_valid.sum()))
print("Within assumed India bounds:",int(coords.coordinates_in_expected_india_region.sum()))
print("Zero-coordinate rows:",int((coords.restaurant_zero_coordinate_flag|coords.delivery_zero_coordinate_flag).sum()))
print("Restaurant statuses:\n",coords.restaurant_coordinates_status.value_counts().to_string())
print("Delivery statuses:\n",coords.delivery_coordinates_status.value_counts().to_string())
neg=coords.restaurant_latitude.lt(0)
print("Negative-latitude examples with separate flags:\n",coords.loc[neg,["restaurant_latitude","restaurant_longitude","restaurant_coordinates_global_range_valid","restaurant_outside_india_region_flag"]].head(5).to_string(index=False))
with_distance=calculate_delivery_distance(coords)
print("Distance count:",int(with_distance.approx_delivery_distance_km.notna().sum()))
print("Invalid pairs assigned distance:",int(with_distance.loc[~with_distance.coordinates_valid,"approx_delivery_distance_km"].notna().sum()))


Global valid pairs: 45584
Within assumed India bounds: 41513
Zero-coordinate rows: 3640
Restaurant statuses:
 restaurant_coordinates_status
valid                            41513
zero_coordinate                   3640
outside_expected_india_region      431
Delivery statuses:
 delivery_coordinates_status
valid                            41944
outside_expected_india_region     3640
Negative-latitude examples with separate flags:
  restaurant_latitude  restaurant_longitude  restaurant_coordinates_global_range_valid  restaurant_outside_india_region_flag
          -27.163303             78.057044                                       True                                  True
          -27.165108             78.015053                                       True                                  True
          -18.551440            -73.804855                                       True                                  True
          -30.366322            -78.070453                              

**Interpretation.** All 45,584 pairs are within global coordinate bounds; 41,513 pass the assumed India-region checks and are used for distance. There are 3,640 zero-coordinate rows and 431 restaurant coordinate records outside the assumed region. Invalid pairs are excluded from distance aggregates, not deleted from the dataset.

## 8. Duplicate handling
No exact duplicate records or duplicate IDs were observed. The reusable helper flags duplicates but retains all rows: repeated identifiers in another extract might represent revisions or distinct events and need an explicit resolution policy.


In [7]:
dup, dup_report=detect_duplicates(normalized)
print(dup_report)
print("Flagged exact rows:",int(dup.exact_duplicate_record_flag.sum()),"| flagged duplicate IDs:",int(dup.duplicate_id_flag.sum()))


{'exact_duplicate_excess_count': 0, 'exact_duplicate_participating_rows': 0, 'duplicate_id_excess_count': 0, 'duplicate_id_participating_rows': 0, 'policy': 'Flag only; do not drop records without a documented resolution rule.'}
Flagged exact rows: 0 | flagged duplicate IDs: 0


## 9. Outlier detection
The pipeline applies Tukey IQR screening (1.5 x IQR) to duration, ratings, age, and multiple-delivery values. An IQR flag is a prompt to inspect a value, not proof of an error. No rows are deleted or values capped. `multiple_deliveries = 3` may be legitimate. The 1-5 rating range is tentative and requires authoritative confirmation.


In [8]:
_, outliers=detect_iqr_outliers(normalized)
for c,r in outliers.items():
    print(f"{c}: Q1={r['q1']}, Q3={r['q3']}, fences=({r['lower_fence']}, {r['upper_fence']}), flags={r['outlier_count']}/{r['valid_count']} ({r['outlier_percent']:.2f}%)")
    print("Examples:",r["examples"][:3])
print("Outside tentative 1-5 rating range:",audit["tentative_rating_range"]["flagged_count"])


time_taken_min: Q1=19.0, Q3=32.0, fences=(-0.5, 51.5), flags=270/45584 (0.59%)
Examples: [{'source_row_index': 34, 'id': '0xb796', 'time_taken_min': 52}, {'source_row_index': 179, 'id': '0x4b5b', 'time_taken_min': 54}, {'source_row_index': 257, 'id': '0xd92f', 'time_taken_min': 53}]
delivery_person_ratings: Q1=4.5, Q3=4.9, fences=(3.8999999999999995, 5.500000000000001), flags=1223/43676 (2.80%)
Examples: [{'source_row_index': 24, 'id': '0x2876', 'delivery_person_ratings': 3.5}, {'source_row_index': 104, 'id': '0x3f01', 'delivery_person_ratings': 3.8}, {'source_row_index': 279, 'id': '0xa053', 'delivery_person_ratings': 3.8}]
delivery_person_age: Q1=25.0, Q3=35.0, fences=(10.0, 50.0), flags=0/43730 (0.00%)
Examples: []
multiple_deliveries: Q1=0.0, Q3=1.0, fences=(-1.5, 2.5), flags=361/44591 (0.81%)
Examples: [{'source_row_index': 0, 'id': '0xcdcd', 'multiple_deliveries': 3.0}, {'source_row_index': 34, 'id': '0xb796', 'multiple_deliveries': 3.0}, {'source_row_index': 150, 'id': '0x87e4',

The source audit flags 270 durations above the 51.5-minute upper fence, 1,223 rating IQR candidates, and 361 multiple-delivery values of 3. The ratings' tentative 1-5 screen flags 53 values of 6; confirm the scale before treating these as invalid.

## 10. Categorical normalization
Whitespace is trimmed and case is canonicalized only for labels matching the observed vocabulary. Missing/blank labels become `Unknown`. Other values remain available for review. The source label `Metropolitian` is preserved; no guessed correction is made.


In [9]:
cats, norm_counts=normalize_categories(normalized)
print("Changed known labels by field:",norm_counts)
print("Raw City frequencies:",source.City.value_counts(dropna=False).to_dict())
print("Processed City frequencies:",cats.city.value_counts(dropna=False).to_dict())


Changed known labels by field: {'weather_conditions': 0, 'road_traffic_density': 0, 'type_of_order': 0, 'type_of_vehicle': 0, 'festival': 0, 'city': 0}
Raw City frequencies: {'Metropolitian': 34087, 'Urban': 10133, nan: 1200, 'Semi-Urban': 164}
Processed City frequencies: {'Metropolitian': 34087, 'Urban': 10133, 'Unknown': 1200, 'Semi-Urban': 164}


## 11. Numeric scaling
No scaling is applied automatically. Original units are interpretable, and tree models generally do not need scaling. If a later scale-sensitive model needs it, split data first, fit scaler statistics on training data only, and apply them to the holdout; fitting on all data leaks information. Do not indiscriminately scale latitude/longitude when using them for geographic distances.

## 12. Run and inspect the pipeline
The next cell reuses the actual preprocessing pipeline in memory and compares it with the saved Week 2 CSV and JSON audit. It writes no files and does not overwrite Week 1.


In [10]:
reproduced=preprocess_dataset(source)
summary=build_audit_summary(source,reproduced)
print("Source shape:",source.shape,"| reproduced:",reproduced.shape,"| saved:",week2_saved.shape)
print("Rows removed:",len(source)-len(reproduced))
print("Source missing:",audit["source"]["missing_by_column"])
print("Processed missing:",audit["processed"]["missing_by_column"])
print("Coordinate results:",audit["coordinates"])
print("Outlier counts:",{k:v["outlier_count"] for k,v in audit["outliers"].items()})
assert len(source)==len(reproduced)==len(week2_saved)==45584
assert int(reproduced.coordinates_valid.sum())==41513
assert reproduced.loc[~reproduced.coordinates_valid,"approx_delivery_distance_km"].isna().all()
assert audit["duplicates"]["exact_duplicate_excess_count"]==0 and audit["duplicates"]["duplicate_id_excess_count"]==0
print("Consistency checks passed.")


Source shape: (45584, 20) | reproduced: (45584, 59) | saved: (45584, 59)
Rows removed: 0
Source missing: {'delivery_person_age': 1854, 'delivery_person_ratings': 1908, 'time_orderd': 1731, 'weather_conditions': 616, 'road_traffic_density': 601, 'multiple_deliveries': 993, 'festival': 228, 'city': 1200}
Processed missing: {'delivery_person_age': 1854, 'delivery_person_ratings': 1908, 'time_orderd': 1731, 'multiple_deliveries': 993, 'order_time_minutes': 2161, 'pickup_time_minutes': 1320, 'order_to_pickup_minutes': 2992, 'order_hour': 2161, 'pickup_hour': 1320, 'approx_delivery_distance_km': 4071, 'delivery_person_ratings_iqr_outlier_flag': 1908, 'delivery_person_age_iqr_outlier_flag': 1854, 'multiple_deliveries_iqr_outlier_flag': 993, 'delivery_person_ratings_outside_tentative_1_5_flag': 1908}
Coordinate results: {'restaurant_status_counts': {'valid': 41513, 'zero_coordinate': 3640, 'outside_expected_india_region': 431}, 'delivery_status_counts': {'valid': 41944, 'outside_expected_india

## 13. Impact on logistics decisions

- Malformed order/pickup times can shift orders into the wrong planning period or distort handoff intervals.
- Invalid coordinates can create false distances and spatial patterns; accepted straight-line distances still do not represent routes.
- Dropping rows with any missing field can change the observed driver/order mix. Imputation may create artificial values.
- Ratings and long durations can shift averages and model evaluation; screening supports review and sensitivity analysis without hiding records.
- Unverified category definitions (especially City, vehicle-condition codes, and multiple-delivery semantics) limit operational conclusions even when values look consistent.

## 14. Conclusion and limitations
The pipeline retains all 45,584 records, parses observed dates, preserves raw values, flags parsing failures, duplicate candidates, coordinate concerns, and IQR outliers, and calculates distance for 41,513 accepted coordinate pairs. It does not automatically delete or cap data. Provenance/license, India bounds, rating range, overnight rollover, and 12-hour interval limit remain unverified or assumption-based. Ambiguous field meanings need authoritative documentation. These are data-quality findings, not causal claims or operational recommendations.
